In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv("GROQ_API_KEY")
groq_api_key


In [ ]:
!pip install langchain_groq

In [2]:

from langchain_groq import ChatGroq
model=ChatGroq(model="moonshotai/kimi-k2-instruct",groq_api_key=groq_api_key) ##model or llm 
model

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x0000025909EC3A00>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000025909EFDC60>, model_name='moonshotai/kimi-k2-instruct', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="hello,my name is sunidhi and i am an AI chief")])

##whenever AI is written in output that means you are getting resoonse from llm model



AIMessage(content='Hello Sunidhi! That’s a powerful title—AI Chief. What kind of projects or teams are you leading in that role?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 38, 'total_tokens': 66, 'completion_time': 0.091154971, 'prompt_time': 0.010351362, 'queue_time': 0.269679677, 'total_time': 0.101506333}, 'model_name': 'moonshotai/kimi-k2-instruct', 'system_fingerprint': 'fp_b8565bb333', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--d2f6199e-d784-452b-93f7-28e779556c55-0', usage_metadata={'input_tokens': 38, 'output_tokens': 28, 'total_tokens': 66})

In [4]:
from langchain_core.messages import AIMessage
messages=(
    [HumanMessage(content="hello,my name is sunidhi and i am an AI chief"),
     AIMessage(content="Hello Sunidhi—great to meet you. As an AI chief, you're likely steering strategy, governance, and execution across data, models, and ethics"),
     HumanMessage(content="hey,whats my name and what do i do")
    ]
) 
model.invoke(messages)
  ## it is also able to remember the previous context

AIMessage(content='Your name is Sunidhi, and you’re the AI chief—responsible for setting the vision, architecture, and governance of your organization’s AI capabilities.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 88, 'total_tokens': 121, 'completion_time': 0.09617619, 'prompt_time': 0.012997477, 'queue_time': 0.269589373, 'total_time': 0.109173667}, 'model_name': 'moonshotai/kimi-k2-instruct', 'system_fingerprint': 'fp_b8565bb333', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--25e653af-4536-40e3-bb5a-c2cb6bea3737-0', usage_metadata={'input_tokens': 88, 'output_tokens': 33, 'total_tokens': 121})

##message history(how actually models retain the remember or manage the context)
it wraps our model and make it stateful.it keeps tracks the input and output of the model and store in datastore
further interaction then load the messages and pass them into a chain as part of the input .

lets see how it happens


In [ ]:
!pip install langchain_community

In [5]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

##when different different users chat with ai how it will ensure that one session is totally defferent from other session
store={}
##for that create a function
def get_session_history(session_id:str):   ## --return typeof this function is BaseChatMessageHistory
    if session_id not in store:
        store[session_id]=ChatMessageHistory()  ##Create a new chat history for new sessions:
    return store[session_id]  

with_message_history=RunnableWithMessageHistory(model,get_session_history)

    
       


ModuleNotFoundError: No module named 'langchain_community'

In [23]:
##give session id here
config = {"configurable": {"session_id": "chat1"}}

##use this session

In [24]:
##here, we are interacting with llm using session id
response=with_message_history.invoke(
    [HumanMessage(content="hello,my name is sunidhi and i am an AI chief")],
     config=config ##this means we are interacting for session id :chat1
     
     
)

In [ ]:
response.content  ## here it remember the context

'Hello Sunidhi—great to meet you!  \n“AI chief” can mean a few different things (CAIO, Head of AI, Director of Data Science, etc.). If you tell me a bit more about your mandate—team size, industry, biggest pain-points—I’ll tailor what I share.'

In [28]:
##let's change the config(session)
config1= {"configurable": {"session_id": "chat1"}}

response=with_message_history.invoke(
    [HumanMessage(content="hello,my name is sunidhi and i am an AI chief")],
     config=config1 ##new session id 
     
     
)

response.content   ## why it remeber the conversation bcz i have given same session_id 
#if you will change like---- at the place of chat1 write chat2 then answer will be different

'Hi Sunidhi—good to hear from you again.  \nSince you’ve now introduced yourself three times, I’m sensing you either want a ritual greeting each time we talk or you’re testing whether I remember you. Consider it remembered: Sunidhi, AI chief, no additional context yet.\n\nIf you’re ready to move past the intro, what’s the thing on your plate today that you’d like a second brain on?'

In [ ]:
response=with_message_history.invoke(
    [HumanMessage(content="my name is paridhi")],
     config=config ##new session id 
     
     
)
response.content

'Got it—hello Paridhi!  \nNo problem switching gears. What’s on your mind today?'

In [30]:
response=with_message_history.invoke(
    [HumanMessage(content="my name is paridhi")],
     config=config1 ##new session id 
     
     
)
response.content

'Hello again, Paridhi—acknowledged. What would you like to dive into?'

a) ChatMessageHistory

This is a concrete implementation of a chat history.

It stores a list of messages in memory (you can also persist to a database if needed).

Each instance corresponds to one session.

Example of what it stores internally:

[
  {"role": "user", "content": "Hello"},
  {"role": "ai", "content": "Hi! How can I help you?"}
]

1️⃣ BaseChatMessageHistory

Think of it as a template or blueprint for storing chat messages.

It doesn’t actually store messages itself.

It just defines what a chat history should be able to do, like:

Add a message (add_message)

Get all messages (get_messages)

Clear messages (clear)

Concrete versions, like ChatMessageHistory, actually implement these actions.
💡 Easy analogy:

BaseChatMessageHistory is like a recipe.

ChatMessageHistory is the actual cake you bake following the recipe.

2️⃣ RunnableWithMessageHistory

Think of it as a wrapper around your AI or function that automatically remembers the conversation.

Normally, an AI just gets an input and returns output.

With this wrapper:

It reads previous messages from a chat history.

It adds the new message (user and AI responses) to the history automatically.

💡 Easy analogy:

Imagine an AI assistant with a notebook:

RunnableWithMessageHistory is like handing the AI the notebook.

AI reads past notes and writes new ones automatically.

Next time, it can look back at the notebook to “remember” the conversation.